In [0]:
import pandas as pd
import numpy as np
from scipy.optimize import linprog
from pyspark.sql import functions as F
 
BATCH_SIZE = 500            # solve for the N most at-risk Category-A items this cycle
BUDGET_HEADROOM_PCT = 0.30  # spend the floor cost, plus 30% of the remaining "nice to have" budget
MAX_BUFFER_MULTIPLIER = 0.5 # upper bound = shortfall + 0.5 x reorder_point

In [0]:
# 1. Deterministic top-N candidate selection (ranked by risk, not an arbitrary row order)
data = spark.table("supply_chain_opt.gold_inventory_master") \
    .filter("priority_level = 'CRITICAL' AND abc_category = 'A'") \
    .orderBy(F.desc("stock_out_risk_score")) \
    .limit(BATCH_SIZE) \
    .toPandas()
 
print(f"Selected {len(data)} of the highest-risk Category-A CRITICAL items for this optimization cycle.")

Selected 500 of the highest-risk Category-A CRITICAL items for this optimization cycle.


In [0]:
# 2. Build bounds: (must order at least the shortfall, don't exceed a sane buffer)
data['shortfall'] = (data['reorder_point_adj'] - data['current_stock']).clip(lower=0)
data['upper_limit'] = data['shortfall'] + MAX_BUFFER_MULTIPLIER * data['reorder_point_adj']
 
bounds = list(zip(data['shortfall'], data['upper_limit']))
 
# 3. Dynamic, data-driven budget (this is what makes the trade-off meaningful:
#    if budget == max_cost, every item just fills to its cap and there's no real decision to make;
#    if budget == min_cost, every item gets exactly its shortfall. Headroom in between is where the
#    solver actually has to choose where the marginal dollar does the most good.)
min_cost = float((data['shortfall'] * data['unit_cost']).sum())
max_cost = float((data['upper_limit'] * data['unit_cost']).sum())
BUDGET = min_cost + BUDGET_HEADROOM_PCT * (max_cost - min_cost)
 
print(f"Floor cost (cover every shortfall): ${min_cost:,.2f}")
print(f"Ceiling cost (fill every item to its buffer cap): ${max_cost:,.2f}")
print(f"This cycle's budget ({int(BUDGET_HEADROOM_PCT*100)}% headroom above floor): ${BUDGET:,.2f}")

Floor cost (cover every shortfall): $110,737,017.48
Ceiling cost (fill every item to its buffer cap): $172,933,784.45
This cycle's budget (30% headroom above floor): $129,396,047.57


In [0]:
# 4. Solve with HiGHS (dual/primal simplex + interior-point, selected automatically by the solver)
#    Objective: maximize total risk-weighted spend, i.e. put the marginal dollar where it reduces
#    the most stock-out risk, subject to the budget.
c = data['stock_out_risk_score'].values * -1
A_ub = [data['unit_cost'].values]
b_ub = [BUDGET]
 
res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
 
if res.success:
    data['optimized_order_qty'] = np.round(res.x)
    print("Optimal order plan found.")
    display(data[['product_name', 'current_stock', 'reorder_point_adj', 'shortfall',
                  'optimized_order_qty', 'unit_cost', 'stock_out_risk_score']])
else:
    print("Solver failed:", res.message)
 

Optimal order plan found.


product_name,current_stock,reorder_point_adj,shortfall,optimized_order_qty,unit_cost,stock_out_risk_score
Item_10822,117,1261,1144,1144.0,496.12,8.44
Item_82978,45,438,393,393.0,450.45,8.4
Item_71694,51,542,491,491.0,505.63,8.32
Item_70021,68,726,658,1021.0,245.65,8.26
Item_71693,104,1139,1035,1035.0,405.1,8.26
Item_71607,31,329,298,298.0,472.07,8.21
Item_77493,69,679,610,950.0,243.04,8.16
Item_69602,104,1136,1032,1600.0,328.9,8.16
Item_95164,50,487,437,680.0,259.98,8.16
Item_13274,84,808,724,724.0,428.07,8.12


In [0]:
 # 5. Sanity check: guard against the old degenerate "everything into one SKU" failure mode
data['spend'] = data['optimized_order_qty'] * data['unit_cost']
max_share = data['spend'].max() / data['spend'].sum()
print(f"Largest single-item share of total spend: {max_share:.1%}")
assert max_share < 0.10, "A single item is absorbing more than 10% of the budget - check the upper bounds."

Largest single-item share of total spend: 0.5%


In [0]:
# 6. Risk reduction: recompute stock_out_risk_score with the new (higher) stock level
data['after_risk_score'] = np.round(
    (data['daily_demand'] * data['lead_time_days']) / (data['current_stock'] + data['optimized_order_qty']), 2
)
 
total_before_risk = data['stock_out_risk_score'].sum()
total_after_risk = data['after_risk_score'].sum()
improvement = (total_before_risk - total_after_risk) / total_before_risk * 100
print(f"Total risk reduction across this batch: {improvement:.2f}%")

Total risk reduction across this batch: 90.51%


In [0]:
import plotly.graph_objects as go
 
# Chart the 20 highest-risk items for readability (the batch itself has 500)
chart_data = data.nlargest(20, 'stock_out_risk_score')
 
fig = go.Figure(data=[
    go.Bar(name='Before Optimization', x=chart_data['product_name'], y=chart_data['stock_out_risk_score'], marker_color='indianred'),
    go.Bar(name='After Optimization', x=chart_data['product_name'], y=chart_data['after_risk_score'], marker_color='lightseagreen')
])
fig.update_layout(
    title='Stock-out Risk: 20 Highest-Risk Items, Before vs After',
    xaxis_title='Product Name',
    yaxis_title='Stock-out Risk Score (lower is better)',
    barmode='group',
    template='plotly_white'
)
fig.show()
 